In [0]:
import pyspark.sql.functions as f
from pyspark.sql.functions import monotonically_increasing_id, concat, lit
from pyspark.sql.window import Window
container_path = "abfss://files-copied@wakadnalearning.dfs.core.windows.net/Silver/"

In [0]:
contents_df = (dbutils.fs.ls(container_path))
file_names =[]
for contents in contents_df:
  file_names.append(contents.name)
print(file_names)


In [0]:
india_df = spark.read.format("delta").load(container_path+file_names[1])
america_df = spark.read.format("delta").load(container_path+file_names[0])
uae_df = spark.read.format("delta").load(container_path+file_names[2])

In [0]:
india_df = india_df.withColumn("unit_price_in_$",
                    f.round(f.col("unit_price") / f.lit(95.7),2)
                    )
                     
uae_df = uae_df.withColumn("unit_price_in_$",
                    f.round(f.col("unit_price") / f.lit(3.67),2)
                    )
america_df = america_df.withColumn("unit_price_in_$",
                    f.round(f.col("unit_price") / f.lit(1),2)
                    )

In [0]:
sales_df = india_df.union(uae_df).union(america_df)
sales_df.printSchema()

In [0]:
# -----------------------------------
# CREATE CUSTOMER DIMENSION
# -----------------------------------
dim_customer = (
    sales_df.select(
        "Customer_Name",
        "City",
        "State",
        "Customer_Email"
    )
    .distinct()
)

# Generate customer_id
window_spec = Window.orderBy("Customer_Name")

dim_customer = dim_customer.withColumn(
    "customer_id",
    concat(
        lit("CUST-"),
        monotonically_increasing_id()
    )
)

# Reorder + rename columns
dim_customer = dim_customer.select(
    "customer_id",
    sales_df.Customer_Name.alias("name"),
    sales_df.City.alias("city"),
    sales_df.State.alias("state"),
    sales_df.Customer_Email.alias("email")
)

# -----------------------------------
# CREATE PRODUCT DIMENSION
# -----------------------------------
dim_product = (
    sales_df.select(
        "Product_Name",
        "Product_Category",
        "Unit_Price",
        "unit_price_in_$"
    )
    .distinct()
)

# Generate product_id
product_window = Window.orderBy("Product_Name")

dim_product = dim_product.withColumn(
    "product_id",
    concat(
        lit("PROD-"),
        monotonically_increasing_id()
    )
)

# Reorder + rename columns
dim_product = dim_product.select(
    "product_id",
    dim_product.Product_Name.alias("name"),
    dim_product.Product_Category.alias("category"),
    dim_product.Unit_Price.alias("unit_price"),
    dim_product["unit_price_in_$"]
)

# -----------------------------------
# CREATEING FACT TABLE
# -----------------------------------

# Join customer dimension
fact_df = sales_df.join(
    dim_customer,
    (
        (sales_df.Customer_Name == dim_customer.name) &
        (sales_df.City == dim_customer.city) &
        (sales_df.State == dim_customer.state) &
        (sales_df.Customer_Email == dim_customer.email)
    ),
    "left"
)

# Join product dimension
fact_df = fact_df.join(
    dim_product,
    (
        (fact_df.Product_Name == dim_product.name) &
        (fact_df.Product_Category == dim_product.category) &
        (fact_df.Unit_Price == dim_product.unit_price) &
        (fact_df["unit_price_in_$"] == dim_product["unit_price_in_$"])
    ),
    "left"
)

# Select fact table columns
main_transactions = fact_df.select(
    fact_df.Transaction_ID.alias("transaction_id"),
    fact_df.customer_id,
    fact_df.Country.alias("country"),
    fact_df.product_id,
    fact_df.Quantity.alias("quantity"),
    fact_df.Payment_Method.alias("payment_type"),
    fact_df.Order_Date.alias("order_date"),
    fact_df.Store_ID.alias("store_id"),
    fact_df.Date_processed
)

In [0]:

main_transactions.show(truncate=False)
dim_customer.show(truncate=False)
dim_product.show(truncate=False)

In [0]:
container_path = "abfss://files-copied@wakadnalearning.dfs.core.windows.net/"
main_transactions.coalesce(1).write.format('csv').option('header', True).mode('overwrite').save(container_path + "Gold/main_transactions_csv")
dim_customer.coalesce(1).write.format('csv').option('header', True).mode('overwrite').save(container_path + "Gold/dim_customer")
dim_product.coalesce(1).write.format('csv').option('header', True).mode('overwrite').save(container_path + "Gold/dim_product")

In [0]:
main_transactions.count()

In [0]:
main_transactions.show(truncate=False)
dim_customer.show(truncate = False)
dim_product.show(truncate = False)